# 5. Stage 5: Data Rejoin

**Stage:** 5.1 of 5 — Daily Unique Rejoin

**Purpose:** Combine all enriched pipeline outputs into one record per vacancy per day. Stage 1 id/region/click data is enriched with the Stage 4.5 geographic lookup and then merged with the Stage 4 ESCO-classified data.

**Input:**
- Stage 4 daily Pickle files from `g.Config.STAGE4_OUTPUT_PATH`
- Stage 1 id/region/click Pickle files referenced by the Stage 5 tracker
- Stage 4.5 region database from `g.Config.STAGE4_5_REGION_DB_PATH`

**Output:** One `ua-YYYY-MM-DD.parquet` file per completed day in `g.Config.STAGE5_DAILY_UNIQUE_PARQUET_PATH`, plus matching full and Ukrainian-language JSON files.

**Run:** Execute all cells from top to bottom. No API calls are required.

## 5.1. Description

The final stage of the processing pipeline consolidates all intermediate outputs into a single, enriched dataset. This unified format allows for structured analysis, filtering, and visualization of the Ukrainian job market across various dimensions such as skills, occupations, regions, and time.

## 5.2. Load packages and configuration

Enable autoreload, add `code/` to the Python path, and trigger memory cleanup.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
from pipeline_bootstrap import configure_pipeline
configure_pipeline()
import general as g
g.clean_memory()

Import `stage1` (for `extract_date_from_file_name`) and `pandas`. The `os` and `pandas` imports are repeated for safety when restarting the kernel mid-session.

In [ ]:
import stage1 as st1
import pandas as pd
import os
import pandas as pd

##  Check folders exists

## 5.3. Get or create process log file

Define `get_stage5_process_df()` — loads or creates the Stage 5 unique process tracker. On first run, it scans the Stage 1 id/region/click folder and registers every available daily file. Subsequent runs load the existing tracker and resume from where processing left off.

In [ ]:
def get_stage5_process_df(stage5_process_path, stage1_region_path):
    if not os.path.exists(stage5_process_path):
        columns = ["input_path", "region_path", "rejoin_path", "rejoin_status"]
        stage5_process = pd.DataFrame(columns=columns).astype(str)
        file_list = [f for f in os.listdir(stage1_region_path) if f]
        for file in file_list:
            path = os.path.join(stage1_region_path, file)
            new_row = pd.DataFrame([{'input_path': g.get_file_name_without_ext(file), 'region_path': path}])
            stage5_process = pd.concat([stage5_process, new_row], ignore_index=True)
        stage5_process.reset_index(drop=True, inplace=True)
        stage5_process.to_pickle(stage5_process_path)
    else:
        stage5_process = pd.read_pickle(stage5_process_path)

    return stage5_process

Initialise or load the process tracker, sort by filename, and preview. The `rejoin_status` column tracks which files have been processed — `"complete"` rows are skipped in the main loop.

In [ ]:
process_df = get_stage5_process_df(g.Config.STAGE5_PROCESS_UNIQUE_PATH, g.Config.STAGE1_ID_REGION_NCLICK_PATH)
process_df = process_df.sort_values(by='input_path').reset_index(drop=True)
process_df

## 5.4. Rejoin data

### 5.4.1. Rejoin unique daily data

In this folder rejoined data with only daily published job vacancies, without prevoius day published

Ensure all three Stage 5 unique output folders exist: PKL, JSON (all languages), and JSON (Ukrainian-only).

In [ ]:
g.check_folder_exists(g.Config.STAGE5_DAILY_UNIQUE_PARQUET_PATH)
g.check_folder_exists(g.Config.STAGE5_DAILY_UNIQUE_JSON_PATH)
g.check_folder_exists(g.Config.STAGE5_DAILY_UNIQUE_JSON_UA_PATH)

g.check_folder_exists(g.Config.STAGE5_MONTHLY_UNIQUE_PARQUET_PATH)
g.check_folder_exists(g.Config.STAGE5_MONTHLY_UNIQUE_JSON_PATH)
g.check_folder_exists(g.Config.STAGE5_MONTHLY_UNIQUE_JSON_UA_PATH)

Load the Stage 4.5 region lookup table (`region_db.pkl`). This DataFrame maps each raw region string (`original`) to its standardised English oblast name, city, district, country, latitude, and longitude. It is used inside the main loop to enrich every vacancy with geographic coordinates.

In [ ]:
region_final_db = pd.read_pickle(g.Config.STAGE4_5_REGION_DB_PATH)
region_final_db

**Main rejoin loop** — processes each daily file in the tracker that has not yet been completed. Three steps per file:

1. **Region enrichment** — merge Stage 1 id/region/click data with `region_final_db` to attach `city`, `district`, `country`, `latitude`, `longitude` and the standardised `region` name
2. **ESCO merge** — join enriched region data with Stage 4 output on `id`
3. **Save** — write result as `.pkl` (full) + `.json` (all languages) + `_ua.json` (Ukrainian descriptions only)

The tracker is updated after each file so the loop can safely be interrupted and resumed.

In [ ]:
for _file, _row in process_df.iterrows():
    if _row["rejoin_status"] == "complete" or not os.path.exists(os.path.join(g.Config.STAGE4_OUTPUT_PATH, f"{_row['input_path']}.pkl")):
        print(f"Day: {_row['input_path']} - already rejoined or Stage 4 output missing, skipping")
        continue

    print(f"Day: {_row['input_path']} - rejoining")

    # --- Step 1: load Stage 1 id/region/click data for this day ---
    id_region_df = pd.read_pickle(process_df.loc[_file, 'region_path'])
    id_region_df['id'] = id_region_df['id'].astype('string')

    # --- Step 2: enrich with lat/lon via region_final_db (Stage 4.5 output) ---
    # Merges raw region string → standardised region, city, district, country, latitude, longitude
    pickle_region = pd.merge(id_region_df, region_final_db,
                             how='left', left_on='region', right_on='original')
    pickle_region.drop(columns=['region_x'], inplace=True)
    pickle_region = pickle_region.rename(columns={'original': 'region_original',
                                                  'region_y': 'region'})
    pickle_region['id'] = pickle_region['id'].astype('string')

    # --- Step 3: merge enriched region data with Stage 4 ESCO-classified output ---
    daily_data = pd.read_pickle(
        os.path.join(g.Config.STAGE4_OUTPUT_PATH, f"{process_df.loc[_file, 'input_path']}.pkl")
    )
    daily_data['id'] = daily_data['id'].astype('string')

    daily_data = pd.merge(pickle_region, daily_data, how='left', on='id')

    # --- Step 4: save outputs ---
    parquet_rejoin_path = os.path.join(g.Config.STAGE5_DAILY_UNIQUE_PARQUET_PATH,
                                   f"{process_df.loc[_file, 'input_path']}.parquet")
    daily_data.to_parquet(parquet_rejoin_path, index=False)

    json_rejoin_path = os.path.join(g.Config.STAGE5_DAILY_UNIQUE_JSON_PATH,
                                    f"{process_df.loc[_file, 'input_path']}.json")
    daily_data.to_json(json_rejoin_path, orient='records')

    # Ukrainian-only export (language filter)
    json_ua_rejoin_path = os.path.join(g.Config.STAGE5_DAILY_UNIQUE_JSON_UA_PATH,
                                       f"{process_df.loc[_file, 'input_path']}.json")
    daily_data[daily_data['desc_lang'] == 'uk'].to_json(json_ua_rejoin_path,
                                                         orient='records', force_ascii=False)

    # --- Step 5: update tracker ---
    process_df.loc[_file, 'rejoin_path'] = parquet_rejoin_path
    process_df.loc[_file, 'rejoin_status'] = "complete"
    process_df.to_pickle(g.Config.STAGE5_PROCESS_UNIQUE_PATH)

    print(f"Day: {process_df.loc[_file, 'input_path']} - done "
          f"({daily_data.shape[0]} rows, {daily_data.shape[1]} cols)")

print("All done.")

## 5.5. Inspection and statistics (optional)

The cells below are diagnostic utilities — run them after the main loop to verify output quality. They are not required for the pipeline to proceed to Stage 5.2.

In [ ]:
pd.read_parquet(os.path.join(g.Config.STAGE5_DAILY_UNIQUE_PARQUET_PATH, "ua-2024-01-01.parquet"))

Reload the tracker and preview the first two rows to confirm `rejoin_status` and `rejoin_path` were written correctly.

In [ ]:
process_df = pd.read_pickle(g.Config.STAGE5_PROCESS_UNIQUE_PATH)
process_df.head(2)

Build a per-file statistics table counting records per language (`en`, `uk`, `ru`, `cs`, `pl`) and how many of those have empty `skill_labels_en`. Useful for spotting extraction gaps across the dataset.

In [ ]:
skill_lebels_en_stats = pd.DataFrame({"rejoin_path": process_df["rejoin_path"],
                                      "en": 0, "uk": 0, "ru":0, "cs": 0, "pl":0,
                                      "en_empty": 0, "uk_empty": 0, "ru_empty":0, "cs_empty": 0, "pl_empty":0})
skill_lebels_en_stats.head(2)

In [ ]:
def get_empty_count(dt, lang):
    sm = (data.loc[data["desc_lang"].eq(lang), "skill_labels_en"].apply(len) < 3).sum()
    return sm

for _, row in skill_lebels_en_stats.iterrows():
    print(f"--- {row["rejoin_path"]} ---")

    path = row.get("rejoin_path", None)

    if pd.isna(path):
        continue

    data = pd.read_parquet(row["rejoin_path"])
    skill_lebels_en_stats.loc[_, "en"] = len(data[data["desc_lang"] == "en"])
    skill_lebels_en_stats.loc[_, "uk"] = len(data[data["desc_lang"] == "uk"])
    skill_lebels_en_stats.loc[_, "ru"] = len(data[data["desc_lang"] == "ru"])
    skill_lebels_en_stats.loc[_, "cs"] = len(data[data["desc_lang"] == "cs"])
    skill_lebels_en_stats.loc[_, "pl"] = len(data[data["desc_lang"] == "pl"])

    skill_lebels_en_stats.loc[_, "en_empty"] = get_empty_count(data, "en")
    skill_lebels_en_stats.loc[_, "uk_empty"] = get_empty_count(data, "uk")
    skill_lebels_en_stats.loc[_, "ru_empty"] = get_empty_count(data, "ru")
    skill_lebels_en_stats.loc[_, "cs_empty"] = get_empty_count(data, "cs")
    skill_lebels_en_stats.loc[_, "pl_empty"] = get_empty_count(data, "pl")

skill_lebels_en_stats

In [ ]:
sms = skill_lebels_en_stats.iloc[:, 1:].sum()
sms

In [ ]:
round(100*sms["en_empty"]/sms["en"],2)

In [ ]:
round(100*sms["pl_empty"]/sms["pl"],2)

In [ ]:
round(100*sms["uk_empty"]/sms["uk"],2)

In [ ]:
round(100*sms["ru_empty"]/sms["ru"],2)

In [ ]:
round(100*sms["cs_empty"]/sms["cs"],2)

## Chesk skill_labesl_en statisics

Aggregate skill coverage statistics by year. For each language and year, counts vacancies with and without English skill labels and plots the distribution as a grouped bar chart.

In [ ]:
import pandas as pd
process_df = pd.read_pickle(g.Config.STAGE5_PROCESS_UNIQUE_PATH)

In [ ]:
stats_df = pd.DataFrame({"year": range(2022,2026), "en": 0, "uk": 0, "ru":0, "cs": 0, "pl":0, "en_total": 0, "uk_total": 0, "ru_total":0, "cs_total": 0, "pl_total":0})
stats_df

In [ ]:
#_ = 0
#row = process_df.iloc[_, :]
langs = ["en", "uk", "ru", "cs", "pl"]
for _, row in process_df.iterrows():
    if not pd.isna(row["rejoin_path"]):
        data = pd.read_parquet(row["rejoin_path"])
        year = pd.to_datetime(data.loc[0, 'date']).year

        for lang in langs:
            stats_df.loc[stats_df["year"] == year, f"{lang}_total"] += data[data["desc_lang"] == lang].shape[0]

        data = data.loc[data["skill_labels_en"].apply(len) < 3]

        for lang in langs:
            stats_df.loc[stats_df["year"] == year, lang] += data[data["desc_lang"] == lang].shape[0]
        print(f"Date: {row["rejoin_path"]} - done")
stats_df

In [ ]:
stats_df.plot.bar(x="year", y=["en", "uk", "ru", "cs", "pl"])

In [ ]:
stats_df_perc = stats_df.copy()

for y in stats_df_perc["year"]:
    for lang in langs:
        stats_df_perc.loc[stats_df["year"] == y, f"{lang}"] = round(100*stats_df_perc.loc[stats_df_perc["year"] == y, f"{lang}"]/stats_df_perc.loc[stats_df_perc["year"] == y, f"{lang}_total"],2)
stats_df_perc

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.